<a href="https://colab.research.google.com/github/AlokDhanush/TB-ResNet50/blob/main/TB_ResNET50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install TF 2.17 (works with Python 3.11 in Colab)
!pip install -q tensorflow==2.17.0

# Install ONNX conversion tools
!pip install -q tf2onnx onnxruntime onnx
!pip install -q kaggle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.4/601.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
jax 0.7.2 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.4.1 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires 

In [6]:
import os
import json

# Ask user for Kaggle token
token = "KGAT_668fe0161dc6aaef93005a85abff42b1"

# Create kaggle.json structure
kaggle_data = {
    "username": "dhanushalok",
    "key": token
}

# Make directory
os.makedirs('/root/.kaggle', exist_ok=True)

# Write kaggle.json file
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_data, f)

# Set correct permissions
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("kaggle.json created successfully!")

kaggle.json created successfully!


In [7]:
# Download and unzip dataset
DATASET_SLUG="tawsifurrahman/tuberculosis-tb-chest-xray-dataset"
!mkdir -p /content/dataset
!kaggle datasets download -d $DATASET_SLUG -p /content/dataset --unzip
# list files to inspect
!ls -la /content/dataset


Dataset URL: https://www.kaggle.com/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset
License(s): copyright-authors
 94% 626M/663M [00:03<00:00, 189MB/s]
100% 663M/663M [00:06<00:00, 100MB/s]
total 12
drwxr-xr-x 3 root root 4096 Dec  6 13:57 .
drwxr-xr-x 1 root root 4096 Dec  6 13:56 ..
drwxr-xr-x 4 root root 4096 Dec  6 13:57 TB_Chest_Radiography_Database


In [8]:
import os, pathlib

DATA_DIR = pathlib.Path("/content/dataset/TB_Chest_Radiography_Database")

# Print a quick tree (first two levels)
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(str(DATA_DIR), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 1:
        # only show top-level content and one level deeper
        for f in files[:10]:
            print(f"{indent}  - {f}")
    if level > 1:
        break

# If your dataset is in a nested folder (e.g., dataset/images/...), you may need to adjust DATA_DIR accordingly.

TB_Chest_Radiography_Database/
  Tuberculosis/
    - Tuberculosis-511.png
    - Tuberculosis-259.png
    - Tuberculosis-194.png
    - Tuberculosis-637.png
    - Tuberculosis-81.png
    - Tuberculosis-568.png
    - Tuberculosis-149.png
    - Tuberculosis-7.png
    - Tuberculosis-586.png
    - Tuberculosis-154.png
  Normal/
    - Normal-3125.png
    - Normal-794.png
    - Normal-826.png
    - Normal-2945.png
    - Normal-2978.png
    - Normal-2797.png
    - Normal-2885.png
    - Normal-19.png
    - Normal-2697.png
    - Normal-2438.png


In [15]:
!pip uninstall -y tensorflow keras numpy ml-dtypes

In [16]:
!pip install tensorflow==2.17.1 numpy==1.26.4

  Using cached tensorflow-2.17.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached ml_dtypes-0.4.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.4/601.4 MB 42.6 MB/s  0:00:14
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached ml_dtypes-0.4.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [tensorflow]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
ja

In [2]:
!pip uninstall -y jax jaxlib ml-dtypes tensorflow keras numpy

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: ml-dtypes 0.4.1
Uninstalling ml-dtypes-0.4.1:
  Successfully uninstalled ml-dtypes-0.4.1
Found existing installation: tensorflow 2.17.1
Uninstalling tensorflow-2.17.1:
  Successfully uninstalled tensorflow-2.17.1
Found existing installation: keras 3.12.0
Uninstalling keras-3.12.0:
  Successfully uninstalled keras-3.12.0
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


In [3]:
!pip install numpy==1.26.4 tensorflow==2.17.1

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached tensorflow-2.17.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached ml_dtypes-0.4.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached keras-3.12.0-py3-none-any.whl.metadata (5.9 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached tensorflow-2.17.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (601.4 MB)
Using cached ml_dtypes-0.4.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
Using cached keras-3.12.0-py3-none-any.whl (1.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [tensorflow]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
orbax-checkpoint 0.11.28 requires j

In [1]:
import tensorflow as tf
import numpy as np
import random
import pathlib
import os

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [2]:
from pathlib import Path
DATA_DIR = Path("/content/dataset/TB_Chest_Radiography_Database")
print("Exists:", DATA_DIR.exists())
for p in DATA_DIR.iterdir():
    if p.is_dir():
        print(p.name, len(list(p.glob('*'))), "files")

Exists: True
Tuberculosis 700 files
Normal 3500 files


In [3]:
import tensorflow as tf
import os, pathlib
SEED = 42
tf.random.set_seed(SEED)

DATA_DIR = pathlib.Path("/content/dataset/TB_Chest_Radiography_Database")
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT = 0.15

# We'll use integer labels (0..C-1) and SparseCategoricalCrossentropy
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='int',
    validation_split=VAL_SPLIT,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='int',
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy(), "Val batches:", tf.data.experimental.cardinality(val_ds).numpy())


Found 4200 files belonging to 2 classes.
Using 3570 files for training.
Found 4200 files belonging to 2 classes.
Using 630 files for validation.
Classes: ['Normal', 'Tuberculosis']
Train batches: 112 Val batches: 20


In [4]:
AUTOTUNE = tf.data.AUTOTUNE

# Cast images to float32 only (we'll let model apply preprocess_input)
def cast_to_float(image, label):
    image = tf.cast(image, tf.float32)
    return image, label

train_ds = train_ds.map(cast_to_float, num_parallel_calls=AUTOTUNE)
val_ds   = val_ds.map(cast_to_float, num_parallel_calls=AUTOTUNE)

# Augmentation pipeline (applied inside the model for best performance usually)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomContrast(0.08),
], name="data_augmentation")

# Prefetch for performance
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [5]:
# Compute counts by scanning directories (fast)
import numpy as np
from collections import Counter

counts = {}
for i, cls in enumerate(class_names):
    counts[i] = len(list((DATA_DIR / cls).glob('*')))
print("Counts per class:", counts)
total = sum(counts.values())
class_weight = {i: total/(NUM_CLASSES * counts[i]) for i in counts}
print("Class weights:", class_weight)

Counts per class: {0: 3500, 1: 700}
Class weights: {0: 0.6, 1: 3.0}


In [6]:
from tensorflow.keras import layers, models, optimizers

base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False

# Build model: augmentation -> preprocess_input -> base -> head
inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32)
x = data_augmentation(inputs)                       # data augmentation only active during training
x = tf.keras.applications.resnet.preprocess_input(x)  # preprocess for ResNet (expects float in 0..255)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ data_augmentation   │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ data_augmentatio… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ data_augmentatio… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ data_augmentatio… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 2048)      │      8,192 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 2)         │        514 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,121,986 (92.02 MB)

 Trainable params: 529,666 (2.02 MB)

 Non-trainable params: 23,592,320 (90.00 MB)

In [7]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard

checkpoint_path = "/content/resnet50_tb_best.h5"
callbacks = [
    ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    TensorBoard(log_dir="/content/logs")
]

EPOCHS_STAGE_1 = 8
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE_1,
    callbacks=callbacks,
    class_weight=class_weight
)

Epoch 1/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.6782 - loss: 0.6757
Epoch 1: val_loss improved from None to 0.08263, saving model to /content/resnet50_tb_best.h5


112/112 ━━━━━━━━━━━━━━━━━━━━ 702s 6s/step - accuracy: 0.7406 - loss: 0.4887 - val_accuracy: 0.9730 - val_loss: 0.0826 - learning_rate: 1.0000e-04
Epoch 2/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.8277 - loss: 0.2773
Epoch 2: val_loss did not improve from 0.08263
112/112 ━━━━━━━━━━━━━━━━━━━━ 594s 5s/step - accuracy: 0.8465 - loss: 0.2684 - val_accuracy: 0.9714 - val_loss: 0.0931 - learning_rate: 1.0000e-04
Epoch 3/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.8659 - loss: 0.2341
Epoch 3: val_loss improved from 0.08263 to 0.07295, saving model to /content/resnet50_tb_best.h5


112/112 ━━━━━━━━━━━━━━━━━━━━ 599s 5s/step - accuracy: 0.8815 - loss: 0.2173 - val_accuracy: 0.9794 - val_loss: 0.0730 - learning_rate: 1.0000e-04
Epoch 4/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.9103 - loss: 0.1642
Epoch 4: val_loss improved from 0.07295 to 0.06977, saving model to /content/resnet50_tb_best.h5


112/112 ━━━━━━━━━━━━━━━━━━━━ 596s 5s/step - accuracy: 0.9132 - loss: 0.1687 - val_accuracy: 0.9794 - val_loss: 0.0698 - learning_rate: 1.0000e-04
Epoch 5/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.9205 - loss: 0.1470
Epoch 5: val_loss improved from 0.06977 to 0.04325, saving model to /content/resnet50_tb_best.h5


112/112 ━━━━━━━━━━━━━━━━━━━━ 629s 5s/step - accuracy: 0.9190 - loss: 0.1476 - val_accuracy: 0.9873 - val_loss: 0.0433 - learning_rate: 1.0000e-04
Epoch 6/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.9281 - loss: 0.1391
Epoch 6: val_loss improved from 0.04325 to 0.03852, saving model to /content/resnet50_tb_best.h5


112/112 ━━━━━━━━━━━━━━━━━━━━ 652s 6s/step - accuracy: 0.9325 - loss: 0.1299 - val_accuracy: 0.9889 - val_loss: 0.0385 - learning_rate: 1.0000e-04
Epoch 7/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.9390 - loss: 0.1389
Epoch 7: val_loss improved from 0.03852 to 0.03052, saving model to /content/resnet50_tb_best.h5


112/112 ━━━━━━━━━━━━━━━━━━━━ 599s 5s/step - accuracy: 0.9437 - loss: 0.1255 - val_accuracy: 0.9921 - val_loss: 0.0305 - learning_rate: 1.0000e-04
Epoch 8/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.9420 - loss: 0.1166
Epoch 8: val_loss improved from 0.03052 to 0.02376, saving model to /content/resnet50_tb_best.h5


112/112 ━━━━━━━━━━━━━━━━━━━━ 646s 6s/step - accuracy: 0.9465 - loss: 0.1087 - val_accuracy: 0.9968 - val_loss: 0.0238 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 8.


In [8]:
# Ensure best weights loaded
model.load_weights(checkpoint_path)

# Evaluate
loss, acc = model.evaluate(val_ds)
print("Validation loss:", loss, "Validation accuracy:", acc)

# Get predictions and compute classification report
import numpy as np
y_true = []
y_pred = []

for X_batch, y_batch in val_ds.unbatch().batch(1000):
    preds = model.predict(X_batch)
    preds_idx = np.argmax(preds, axis=1)
    y_pred.extend(preds_idx.tolist())
    y_true.extend(y_batch.numpy().tolist())

from sklearn.metrics import classification_report, confusion_matrix
print("Classification report:")
print(classification_report(y_true, y_pred, target_names=class_names))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

20/20 ━━━━━━━━━━━━━━━━━━━━ 94s 4s/step - accuracy: 0.9968 - loss: 0.0238
Validation loss: 0.023757947608828545 Validation accuracy: 0.9968253970146179
20/20 ━━━━━━━━━━━━━━━━━━━━ 95s 5s/step
Classification report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       527
Tuberculosis       0.98      1.00      0.99       103

    accuracy                           1.00       630
   macro avg       0.99      1.00      0.99       630
weighted avg       1.00      1.00      1.00       630

Confusion matrix:
[[525   2]
 [  0 103]]


In [18]:
# Save best weights as H5 (legacy)
model.save("/content/tb_resnet50_best.h5")
print("Saved: /content/tb_resnet50_best.h5")

# Save native Keras format (recommended)
model.save("/content/tb_resnet50_best.keras")
print("Saved: /content/tb_resnet50_best.keras")


Saved: /content/tb_resnet50_best.h5
Saved: /content/tb_resnet50_best.keras


In [12]:
!pip install tf2onnx

import tensorflow as tf
import tf2onnx

onnx_path = "/content/tb_resnet50.onnx"
!python -m tf2onnx.convert \
    --saved-model /content/tf_saved_model \
    --output $onnx_path \
    --opset 13

print("ONNX model saved to:", onnx_path)

<frozen runpy>:128: RuntimeWarning: 'tf2onnx.convert' found in sys.modules after import of package 'tf2onnx', but prior to execution of 'tf2onnx.convert'; this may result in unpredictable behaviour
2025-12-06 15:46:59,679 - WARNING - ***IMPORTANT*** Installed protobuf is not cpp accelerated. Conversion will be extremely slow. See https://github.com/onnx/tensorflow-onnx/issues/1557
2025-12-06 15:46:59,795 - WARNING - '--tag' not specified for saved_model. Using --tag serve
2025-12-06 15:47:06,463 - INFO - Signatures found in model: [serve,serving_default].
2025-12-06 15:47:06,463 - WARNING - '--signature_def' not specified, using first signature: serve
2025-12-06 15:47:06,464 - INFO - Output names: ['output_0']
I0000 00:00:1765036026.857464   32605 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1765036029.348564   32605 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2025-12-06 15:47:09,901 - WARN

In [14]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/tb_resnet50_best.h5 /content/drive/MyDrive/
!cp /content/tb_resnet50.onnx /content/drive/MyDrive/
!cp -r /content/tf_saved_model /content/drive/MyDrive/
print("Saved model files to Google Drive (MyDrive).")

Mounted at /content/drive
cp: cannot stat '/content/tb_resnet50_best.h5': No such file or directory
Saved model files to Google Drive (MyDrive).


In [15]:
!pip install onnxruntime

In [19]:
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
import tensorflow as tf

# Load your trained model
model = tf.keras.models.load_model("/content/tb_resnet50_best.keras")

# Check if first layer is DataAugmentation
if isinstance(model.layers[0], tf.keras.layers.RandomFlip) or \
   isinstance(model.layers[0], tf.keras.layers.RandomRotation):
    # Remove the augmentation layer for inference
    inputs = tf.keras.Input(shape=(224,224,3))
    x = model.layers[1](inputs)  # skip first augmentation layer
    for layer in model.layers[2:]:
        x = layer(x)
    model = Model(inputs, x)
    print("Removed augmentation layers for inference")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [21]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input

# Load your trained Keras model
model = tf.keras.models.load_model("/content/tb_resnet50_best.keras")

# Remove augmentation layers for inference
# Assume first layer is augmentation
if isinstance(model.layers[0], tf.keras.layers.RandomFlip) or \
   isinstance(model.layers[0], tf.keras.layers.RandomRotation) or \
   isinstance(model.layers[0], tf.keras.layers.RandomZoom):

    inputs = Input(shape=(224,224,3), name="input_1")
    x = inputs
    for layer in model.layers[1:]:
        x = layer(x)
    model = Model(inputs, x)
    print("Removed augmentation layers for ONNX export")

# Verify model summary
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ data_augmentation   │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ data_augmentatio… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ data_augmentatio… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ data_augmentatio… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 2048)      │      8,192 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 2)         │        514 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,651,654 (94.04 MB)

 Trainable params: 529,666 (2.02 MB)

 Non-trainable params: 23,592,320 (90.00 MB)

 Optimizer params: 529,668 (2.02 MB)

In [22]:
import numpy as np

dummy_input = np.random.randn(1,224,224,3).astype(np.float32)

In [23]:
!pip install tf2onnx --quiet

import tf2onnx

onnx_path = "/content/tb_resnet50_clean.onnx"

# Export
model_proto, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=(tf.TensorSpec([None, 224, 224, 3], tf.float32, name="input_1"),),
    opset=13,
    output_path=onnx_path
)

print("ONNX model saved at", onnx_path)

ONNX model saved at /content/tb_resnet50_clean.onnx


In [24]:
import onnxruntime as ort

session = ort.InferenceSession(onnx_path)

print("ONNX model loaded!")


ONNX model loaded!


In [25]:
class_names = ["Normal", "Tuberculosis"]


In [26]:
import numpy as np
import cv2
import tensorflow as tf  # only for preprocessing function, not for inference

IMG_SIZE = (224, 224)

def preprocess_for_onnx(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE)

    # Convert to float32
    img = img.astype(np.float32)

    # Use the same preprocessing as ResNet50 training
    img = tf.keras.applications.resnet50.preprocess_input(img)

    # Add batch dimension
    img = np.expand_dims(img, axis=0)

    return img

In [27]:
image_path = "/content/tb.jpg"   # <-- put your image here

input_tensor = preprocess_for_onnx(image_path)

# Get input & output names
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# Run inference
pred = session.run([output_name], {input_name: input_tensor})[0][0]
predicted_index = np.argmax(pred)
predicted_label = class_names[predicted_index]

print("Prediction:", predicted_label)
print("Raw probabilities:", pred)

Prediction: Tuberculosis
Raw probabilities: [0.0660701 0.9339299]


In [28]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/tb_resnet50_clean.onnx /content/drive/MyDrive/
print("Saved model files to Google Drive (MyDrive).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved model files to Google Drive (MyDrive).
